# Backtest: does the pre-2025 frontier trend predict the 2025 frontier?

Using ability estimates from the flagship K=4 fit, this notebook fits the frontier trend (per posterior draw, ordinary least squares of ability on release date) on record-setters released before 2025-01-01, extrapolates forward, and compares the prediction against the post-cutoff (2025-2026) record-setters' abilities from the same fit.

**Result.** The pre-2025 trend overpredicts the later frontier on the two well-measured axes. On axis 1 the post-cutoff record-setters land a mean 0.33 logits below the extrapolated line (-1.0 posterior SD; 4/11 inside the predicted 50% band). On axis 2 the trend holds through 2025 (8/8 inside) and breaks in 2026 (1/3 inside; the 2026 records sit 0.3-0.5 below the line). Axes 3 and 4 had only two pre-2025 records each; their bands are too wide to falsify (12/12 inside).

## What this tests, and what it does not

The trend line never sees a post-cutoff point, so the extrapolation is honestly out of sample. The post-cutoff models' ability estimates come from the full fit: their benchmark scores informed the posterior. The precision filters (sd_cap, low-obs, SOTA list) were also computed on the full data. This tests whether the pre-2025 trend predicts the later frontier, not whether the model would place unseen models correctly.

A true leave-2025-out test of the measurement layer requires a refit with the loader's `max_release_date` option set to the cutoff. This notebook does not run that test.

## Setup

Load the flagship posterior lazily: ability and loading draws only (~3 GB of the 18 GB file). Rebuild the matching data. Take theta in the fit's canonical frame (the flagship's display frame is the raw rank-tracked one, no rotation).

In [1]:
import sys
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xarray as xr

# The notebook lives in 3_diagnostics/; repo modules and relative paths need the root.
ROOT = Path.cwd().parent if Path.cwd().name == "diagnostics" else Path.cwd()
sys.path.insert(0, str(ROOT))

import config
from analysis.fitview import prepare_fit
from analysis.factors import trace_axis_names
from analysis.forecast import mirt_frontier_forecast, mirt_crossover_df, _to_year
from analysis.timelines import mirt_model_timeline_df, mirt_human_axis_stats
from viz.forecast import capability_forecast_fig
from diagnostics.diagnose_chains import _load_matching_data

CUTOFF = pd.Timestamp("2025-01-01")
FIT_START = "2024-01-01"   # dashboard uses 2024-10-01; that leaves 3 months before this cutoff
SD_CAP = 0.4
HDI = 0.5
K = 4

In [2]:
TRACE = (ROOT / "results/mirt_humanmerge_lineageprior_lineagebm_dropFrontierMathv1AlgoTune_floors_poolednoise"
         / "trace_mirt_k4_humanmerge_lineageprior_lineagebm_dropFrontierMathv1AlgoTune_floors_poolednoise.nc")

# The trace is 18 GB; open the posterior group lazily and keep only the three
# variables the canonicalization reads. az.from_netcdf would pull everything.
post_ds = xr.open_dataset(TRACE, group="posterior")
sub = post_ds[["A", "theta", "tau_A"]]
sub.attrs.update(post_ds.attrs)
idata = az.InferenceData(posterior=sub)
print(f"{idata.posterior.sizes['chain']} chains x {idata.posterior.sizes['draw']} draws, "
      f"rotation={idata.posterior.attrs.get('mirt_display_rotation')!r}")

10 chains x 10000 draws, rotation='none'


In [3]:
data = _load_matching_data(idata)
raw_df = pd.read_csv(ROOT / "data" / "processed" / "benchmarks_merged.csv")
# Draw the projected line over the whole actual range, not to a fixed date.
HORIZON = pd.to_datetime(raw_df["release_date"], errors="coerce").max() + pd.DateOffset(months=2)
print(f"{data.n_models} models x {data.n_benchmarks} benchmarks, {data.n_obs} obs, "
      f"horizon {HORIZON.date()}")

   drop_benchmarks: removed ['AlgoTune', 'FrontierMath v1'] (119 obs); 98 benchmarks remain
835 models x 98 benchmarks, 5004 obs, horizon 2026-10-02


In [4]:
view = prepare_fit(idata, data)
theta = view.theta                                   # (S, M, K)
axis_names = trace_axis_names(idata, K)
model_names = data.mlookup.sort_values("model_idx")["model"].tolist()
print(theta.shape, axis_names, f"rotated={view.rotated}")

(100000, 835, 4) ['axis1', 'axis2', 'axis3', 'axis4'] rotated=False


In [5]:
# Frame check: posterior-mean theta must match the flagship's saved factor
# scores, else the axes here are not the published axes.
fs = pd.read_csv(TRACE.parent / "mirt_factor_scores.csv").set_index("model")
cols = [f"theta{k + 1}_mean" for k in range(K)]
mine = pd.DataFrame(theta.mean(0), index=model_names, columns=cols)
shared = fs.index.intersection(mine.index)
assert len(shared) > 700 and np.allclose(fs.loc[shared, cols], mine.loc[shared], atol=1e-3)
print(f"frame check passed on {len(shared)} models")

frame check passed on 835 models


## Trend fitted on pre-2025 records

Filter the data frame to rows released before 2025-01-01. Prune the curated release-date table to the same cutoff: the date helper fills missing dates from that table and would silently re-inject the post-cutoff models. Then run the same forecast call the dashboard uses (records basis, 50% intervals), per axis.

In [6]:
DATES_FULL = config.RELEASE_DATES
rd = pd.to_datetime(raw_df["release_date"], errors="coerce")
raw_past = raw_df[(rd < CUTOFF) | rd.isna()]         # undated rows kept
print(f"{len(raw_past)} / {len(raw_df)} rows pre-cutoff or undated")

2354 / 5064 rows pre-cutoff or undated


In [7]:
# The leak: _release_dates fills dates from config.RELEASE_DATES for any model
# absent from the frame, which is exactly the filtered-out 2025+ models.
config.RELEASE_DATES = {m: d for m, d in DATES_FULL.items() if pd.Timestamp(d) < CUTOFF}

fcs, tl_past, hstats, cxs = {}, {}, {}, {}
for k in range(K):
    tl_past[k] = mirt_model_timeline_df(theta, k, data, raw_past, sd_cap=SD_CAP, hdi_prob=HDI)
    fcs[k] = mirt_frontier_forecast(
        theta, k, data, raw_past, fit_basis="records", fit_start=FIT_START,
        sd_cap=SD_CAP, sota_exempt=False, back_start=tl_past[k]["release_date"].min(),
        horizon_date=HORIZON, hdi_prob=HDI)
    hstats[k] = mirt_human_axis_stats(theta, k, data, hdi_prob=HDI)
    cxs[k] = mirt_crossover_df(fcs[k], theta, k, data, axis_name=axis_names[k], hdi_prob=HDI)
    print(f"{axis_names[k]}: trend fit on {len(fcs[k].fit_names)} records")

config.RELEASE_DATES = DATES_FULL

axis1: trend fit on 5 records


axis2: trend fit on 9 records


axis3: trend fit on 2 records


axis4: trend fit on 2 records


## The 2025-2026 actuals

Restore the dates and take the post-cutoff models from the full timeline. Per axis, flag the record-setters: the running record among post-cutoff models, seeded at the pre-2025 record level.

In [8]:
tl_full, actual = {}, {}
for k in range(K):
    tl_full[k] = mirt_model_timeline_df(theta, k, data, raw_df, sd_cap=SD_CAP, hdi_prob=HDI)
    rows = tl_full[k][tl_full[k]["release_date"] >= CUTOFF].sort_values("release_date")
    seed = tl_past[k].loc[tl_past[k]["name"].isin(fcs[k].frontier_names), "mean"].max()
    prev_best = np.maximum.accumulate(np.concatenate([[seed], rows["mean"].values[:-1]]))
    actual[k] = rows.assign(is_record=rows["mean"].values > prev_best).reset_index(drop=True)
    print(f"{axis_names[k]}: {len(actual[k])} post-cutoff models, "
          f"{int(actual[k]['is_record'].sum())} records")

axis1: 257 post-cutoff models, 11 records


axis2: 177 post-cutoff models, 11 records


axis3: 85 post-cutoff models, 10 records


axis4: 83 post-cutoff models, 2 records


## Predicted vs actual

For each post-cutoff record-setter, evaluate the per-draw trend line at its release date and compare with its per-draw ability. Errors are in units of the model's posterior standard deviation. Coverage is the share of records whose actual median falls inside the predicted central 50% band; a calibrated band holds about half.

In [9]:
cmp = {}
for k in range(K):
    rows = []
    for _, r in actual[k][actual[k]["is_record"]].iterrows():
        i = model_names.index(r["name"])
        t = float(_to_year(r["release_date"]))
        pred = fcs[k].intercept + fcs[k].slope * t       # (S,)
        act = theta[:, i, k]                             # (S,)
        p_lo, p_med, p_hi = np.quantile(pred, [0.25, 0.5, 0.75])
        a_lo, a_med, a_hi = np.quantile(act, [0.25, 0.5, 0.75])
        rows.append({
            "model": r["name"], "release_date": r["release_date"].date(),
            "pred_med": p_med, "pred_lo": p_lo, "pred_hi": p_hi,
            "actual_med": a_med, "actual_lo": a_lo, "actual_hi": a_hi,
            "error": a_med - p_med, "error_sd": (a_med - p_med) / act.std(),
            "inside": p_lo <= a_med <= p_hi})
    cmp[k] = pd.DataFrame(rows)

In [10]:
summary = pd.DataFrame([{
    "axis": axis_names[k],
    "n_records": len(cmp[k]),
    "coverage": cmp[k]["inside"].mean(),
    "bias": cmp[k]["error"].mean(),
    "bias_sd": cmp[k]["error_sd"].mean()}
    for k in range(K) if len(cmp[k])])

In [11]:
for k in range(K):
    if len(cmp[k]):
        print(axis_names[k])
        display(cmp[k].round(3))

axis1


,model,release_date,pred_med,pred_lo,pred_hi,actual_med,actual_lo,actual_hi,error,error_sd,inside
0,o3-mini-2025-01-31_high,2025-01-31,0.642,0.503,0.787,0.681,0.561,0.805,0.039,0.208,True
1,o3-2025-04-16_unknown,2025-04-16,0.916,0.731,1.103,0.753,0.596,0.910,-0.163,-0.687,True
2,gemini-2.5-pro-preview-06-05,2025-06-05,1.096,0.880,1.316,0.783,0.656,0.909,-0.313,-1.660,False
3,grok-4-0709,2025-07-09,1.219,0.980,1.461,1.062,0.945,1.180,-0.157,-0.886,True
4,qwen3-max-2025-09-23,2025-09-24,1.496,1.205,1.791,1.148,0.962,1.338,-0.348,-1.162,False
5,gpt-5.1-codex-max,2025-11-19,1.697,1.368,2.033,1.337,1.101,1.585,-0.360,-0.972,False
6,gpt-5.2-codex,2025-12-18,1.802,1.453,2.158,1.410,1.188,1.646,-0.392,-1.099,False
7,gpt-5.3-codex,2026-02-05,1.978,1.594,2.369,1.652,1.409,1.904,-0.326,-0.860,True
8,muse-spark,2026-04-08,2.202,1.773,2.638,1.725,1.495,1.940,-0.477,-1.194,False
9,gpt-5.5-codex,2026-04-23,2.256,1.817,2.703,1.755,1.457,2.060,-0.501,-1.107,False


axis2


,model,release_date,pred_med,pred_lo,pred_hi,actual_med,actual_lo,actual_hi,error,error_sd,inside
0,o3-mini-2025-01-31_high,2025-01-31,1.084,0.910,1.242,0.980,0.831,1.114,-0.104,-0.297,True
1,gemini-2.5-pro-exp-03-25,2025-03-25,1.205,1.007,1.384,1.036,0.846,1.211,-0.169,-0.448,True
2,o3-2025-04-16_high,2025-04-16,1.255,1.047,1.444,1.064,0.910,1.206,-0.191,-0.709,True
3,o4-mini-2025-04-16_high,2025-04-16,1.255,1.047,1.444,1.170,1.034,1.290,-0.085,-0.295,True
4,o3-pro-2025-06-10_high,2025-06-10,1.381,1.145,1.596,1.234,0.802,1.640,-0.147,-0.242,True
5,gpt-5-mini-2025-08-07_high,2025-08-07,1.516,1.246,1.758,1.432,1.285,1.562,-0.084,-0.239,True
6,gpt-5-2025-08-07_high,2025-08-07,1.516,1.246,1.758,1.492,1.348,1.622,-0.023,-0.076,True
7,gpt-5-pro-2025-10-06_unknown,2025-10-07,1.656,1.349,1.931,1.577,1.378,1.764,-0.079,-0.235,True
8,gpt-5.4-pro-2026-03-05_xhigh,2026-03-05,1.999,1.597,2.356,1.586,1.428,1.742,-0.413,-1.640,False
9,gpt-5.5-pre-release_xhigh,2026-04-23,2.112,1.678,2.497,1.621,1.464,1.781,-0.491,-1.841,False


axis3


,model,release_date,pred_med,pred_lo,pred_hi,actual_med,actual_lo,actual_hi,error,error_sd,inside
0,claude-opus-4-20250514_16K,2025-05-22,1.285,0.827,1.703,1.035,0.818,1.256,-0.250,-0.777,True
1,claude-opus-4-20250514,2025-05-22,1.285,0.827,1.703,1.046,0.827,1.284,-0.239,-0.754,True
2,claude-opus-4-1-20250805_16K,2025-08-05,1.393,0.846,1.904,1.080,0.865,1.303,-0.313,-0.964,True
3,claude-opus-4-1-20250805,2025-08-05,1.393,0.846,1.904,1.272,1.075,1.459,-0.121,-0.407,True
4,gpt-5-pro-2025-10-06_unknown,2025-10-07,1.486,0.856,2.074,1.308,1.049,1.545,-0.178,-0.471,True
5,claude-opus-4-5-20251101_16K,2025-11-24,1.555,0.860,2.206,1.318,1.037,1.575,-0.237,-0.607,True
6,claude-opus-4-6,2026-02-05,1.659,0.867,2.409,1.746,0.875,2.092,0.087,0.126,True
7,claude-opus-4-6_high,2026-02-05,1.659,0.867,2.409,1.760,0.883,2.137,0.100,0.141,True
8,claude-opus-4-8_unknown,2026-05-28,1.822,0.875,2.723,2.114,0.725,2.466,0.292,0.315,True
9,claude-fable-5,2026-06-09,1.839,0.875,2.757,2.480,0.855,2.904,0.641,0.586,True


axis4


,model,release_date,pred_med,pred_lo,pred_hi,actual_med,actual_lo,actual_hi,error,error_sd,inside
0,gemini-2.5-pro-exp-03-25,2025-03-25,1.415,1.063,1.746,1.405,0.080,1.641,-0.009,-0.011,True
1,o3-pro-2025-06-10,2025-06-10,1.522,1.087,1.941,1.539,0.727,2.049,0.017,0.019,True


In [12]:
display(summary.round(3))

,axis,n_records,coverage,bias,bias_sd
0,axis1,11,0.364,-0.328,-0.998
1,axis2,11,0.818,-0.193,-0.658
2,axis3,10,1.000,-0.022,-0.281
3,axis4,2,1.000,0.004,0.004


## Figures

Per axis: pre-cutoff points, trend band, human bands, and the post-cutoff points overlaid (filled diamonds = record-setters, open circles = the rest).

In [13]:
for k in range(K):
    fig = capability_forecast_fig(tl_past[k], hstats[k], fcs[k], cxs[k], axis_name=axis_names[k])
    fig.add_vline(x=CUTOFF.strftime("%Y-%m-%d"),
                  line=dict(color="#666", width=1.5),
                  annotation_text="cutoff", annotation_position="top",
                  annotation_font_size=9, annotation_font_color="#666")

    for mask, name, marker in [
            (actual[k]["is_record"], "2025+ record",
             dict(color="#1f77b4", size=10, symbol="diamond")),
            (~actual[k]["is_record"], "2025+ other",
             dict(symbol="circle-open", size=8, color="#1f77b4",
                  line=dict(width=1.5, color="#1f77b4")))]:
        pts = actual[k][mask]
        if len(pts):
            fig.add_trace(go.Scatter(
                x=pts["release_date"].dt.strftime("%Y-%m-%d"), y=pts["mean"],
                mode="markers", marker=marker, name=name, text=pts["name"],
                error_y=dict(type="data", symmetric=False,
                             array=(pts["hdi_high"] - pts["mean"]).values,
                             arrayminus=(pts["mean"] - pts["hdi_low"]).values,
                             color="#1f77b4", thickness=1.2, width=3),
                hovertemplate="<b>%{text}</b><br>%{y:.2f}<br>%{x}<extra></extra>"))

    # Re-pin both axes so the 2025 points are inside the frame; y stays pinned
    # to data, never to the forecast band (same rule as the base figure).
    xmax = max(pd.to_datetime(fcs[k].grid_dates).max(), actual[k]["release_date"].max())
    fig.update_xaxes(range=[tl_past[k]["release_date"].min().strftime("%Y-%m-%d"),
                            xmax.strftime("%Y-%m-%d")])
    ylo = min(tl_past[k]["hdi_low"].min(), actual[k]["hdi_low"].min(), hstats[k]["hdi_low"].min())
    yhi = max(tl_past[k]["hdi_high"].max(), actual[k]["hdi_high"].max(), hstats[k]["hdi_high"].max())
    pad = 0.06 * (yhi - ylo)
    fig.update_yaxes(range=[ylo - pad, yhi + pad])
    fig.update_layout(title=dict(
        text=f"{axis_names[k]} — trend fit on pre-2025 records, 2025-2026 actuals overlaid", x=0.5))
    fig.show()

In [14]:
assert config.RELEASE_DATES == DATES_FULL, "config.RELEASE_DATES not restored"